# Phase 3b eval — score a trained LoRA adapter against held-out val

Runs the same 169 val rows with the same prompts/sampling as Phase 1 baseline (71.60%) so the comparison is apples-to-apples. Only the adapter differs.

**Run this in a SEPARATE Colab session from training** — vLLM and Unsloth have different version pins that conflict if installed in the same kernel.

**Prompt consistency**: the `SYSTEM_PROMPT_MATH` / `SYSTEM_PROMPT_MCQ` strings in section 4 below MUST match `EXPECTED_SYS_MATH` / `EXPECTED_SYS_MCQ` in `train_sft.ipynb` section 5, character-for-character. If you change one, change the other. Section 4 here asserts the training data used the same prompts before generating anything.

To eval all four epoch checkpoints, change `ADAPTER_PATH` in section 3 and re-run sections 3–7 for each.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/third_try'
OUTPUT_DIR  = f'{PROJECT_DIR}/outputs'
import sys
sys.path.insert(0, PROJECT_DIR)
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

Mounted at /content/drive


## 1. Install vLLM + grader deps

Same coherent pin set as Phase 1 (vllm 0.9.2 + torch 2.7.0 + transformers 4.53.3). After this, **Runtime → Restart**, then run from section 2.

In [ ]:
!pip install -q uv 2>&1 | tail -1
!uv pip install --system torch==2.7.0 --index-url https://download.pytorch.org/whl/cu126
!uv pip install --system "vllm==0.9.2" "transformers==4.53.3" \
    sympy "antlr4-python3-runtime==4.11.1"
print("Install done. RESTART THE RUNTIME, then run from section 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 96.9 MB/s eta 0:00:00
Using Python 3.12.13 environment at: /usr
Resolved 25 packages in 1.76s
Prepared 16 packages in 26.17s
Uninstalled 16 packages in 871ms
Installed 16 packages in 181ms
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.6.4.1
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.6.80
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.6.77
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.6.77
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.3.0.4
 - nvidia-cufile-cu12==1.13.1.3
 + nvidia-cufile-cu12==1.11.1.6
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.7.77
 - nvidia-cusolver-cu12==11.7.3.90
 + nvidia-cusolver-cu12==11.7.1.2
 - nvidia-cusparse-cu12==12.5.8.93
 + nvidia-cusparse-cu12==12.5.4.2
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0.6.3
 - nvidia-nccl-c

## 2. Post-restart engine flags + version sanity

In [ ]:
import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
# Uncomment if engine init fails — surfaces the real exception:
# os.environ['VLLM_ENABLE_V1_MULTIPROCESSING'] = '0'

import sys
PROJECT_DIR = f'/content/drive/MyDrive/third_try'
OUTPUT_DIR  = f'{PROJECT_DIR}/outputs'
sys.path.insert(0, PROJECT_DIR)

import torch, transformers, vllm
print('torch       :', torch.__version__)
print('transformers:', transformers.__version__)
print('vllm        :', vllm.__version__)
print('device      :', torch.cuda.get_device_name(0))
print('GPU free    :', round(torch.cuda.mem_get_info(0)[0] / 1e9, 2), 'GB')

assert transformers.__version__ == '4.53.3', 'wrong transformers — restart runtime'

torch       : 2.7.0+cu126
transformers: 4.53.3
vllm        : 0.9.2
device      : NVIDIA A100-SXM4-40GB
GPU free    : 41.96 GB


## 3. Config — change ADAPTER_PATH per checkpoint you want to eval

In [ ]:
ADAPTER_PATH    = f'{PROJECT_DIR}/qwen3_4b_thinking_sft_lora'   # <-- set to actual checkpoint dir name

MODEL_ID        = 'Qwen/Qwen3-4B-Thinking-2507'    # base; adapter applies on top
DATA_PATH       = f'{PROJECT_DIR}/public.jsonl'
VAL_IDS_PATH    = f'{PROJECT_DIR}/val_ids.json'
SFT_DATA_PATH   = f'{PROJECT_DIR}/sft_openr1.jsonl'  # used only for the prompt-match assertion in §4

MAX_GEN_TOKENS  = 32768
MAX_MODEL_LEN   = 40960
GPU_MEM_UTIL    = 0.85
SEED            = 151

import os
assert os.path.isdir(ADAPTER_PATH), f'adapter not found: {ADAPTER_PATH}'
print(f'evaluating adapter: {ADAPTER_PATH}')

evaluating adapter: /content/drive/MyDrive/third_try/qwen3_4b_thinking_sft_lora


## 4. Prompts + apples-to-apples assertion

These prompts produced the Phase-1 baseline (71.60% MC 80.4 / FS 66.7 / FM 67.7). The assertion below confirms the SFT data also used these — otherwise the adapter was trained under a different prompt regime and the lift number is contaminated by distribution shift, not just SFT quality.

In [ ]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Give your final answer inside a single \\boxed{}. "
    "Use EXACT values: prefer fractions (\\frac{a}{b}) and symbolic forms "
    "(\\sqrt{}, \\pi, e) over decimals. If you must give a decimal, write at "
    "least 10 significant figures and do NOT round. "
    "If the problem has multiple sub-answers, put them all inside one \\boxed{}, "
    "comma-separated, in the order asked, e.g. \\boxed{41, 35, 16}. "
    "If a single sub-answer itself contains a comma (a point or tuple), wrap it "
    "in parentheses, e.g. \\boxed{(2, 3), 7}."
)
SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. Read the problem and the answer choices, "
    "then select the single best answer. After your reasoning, output ONLY the "
    "letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}. "
    "The very last thing in your response must be that \\boxed{<letter>}."
)

# Apples-to-apples assertion: verify SFT data used these exact prompts.
import json, os
if os.path.exists(SFT_DATA_PATH):
    sft_rows = [json.loads(l) for l in open(SFT_DATA_PATH)]
    sft_prompts = set(r['system'] for r in sft_rows)
    extra = sft_prompts - {SYSTEM_PROMPT_MATH, SYSTEM_PROMPT_MCQ}
    if extra:
        sample = next(iter(extra))
        raise AssertionError(
            f'SFT data contains system prompts that do NOT match the eval prompts. '
            f'The adapter was trained under a different prompt regime than what '
            f'this eval will send.\nOffending prompt preview:\n  {sample[:200]!r}'
        )
    print(f'apples-to-apples: OK (SFT data uses the same prompts as eval)')
else:
    print(f'  (SFT data file not present at {SFT_DATA_PATH}; skipping cross-check)')

def build_chat(question, options):
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts = "\n".join(f"{l}. {o.strip()}" for l, o in zip(labels, options))
        return [{"role": "system", "content": SYSTEM_PROMPT_MCQ},
                {"role": "user", "content": f"{question}\n\nOptions:\n{opts}"}]
    return [{"role": "system", "content": SYSTEM_PROMPT_MATH},
            {"role": "user", "content": question}]

apples-to-apples: OK (SFT data uses the same prompts as eval)


## 5. Load model + adapter, build prompts

In [ ]:
import json
import harness as H
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

data = H.load_jsonl(DATA_PATH)
val_ids = set(json.load(open(VAL_IDS_PATH)))
val_rows = [r for r in data if r['id'] in val_ids]
print(f'VAL: {len(val_rows)} held-out rows')

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
prompts, order = [], []
for r in val_rows:
    msgs = build_chat(r['question'], r.get('options'))
    prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    order.append(r['id'])

plens = [len(tok(p).input_ids) for p in prompts]
print(f'prompt tokens: max={max(plens)} | headroom for gen = {MAX_MODEL_LEN - max(plens)}')
assert MAX_MODEL_LEN - max(plens) >= MAX_GEN_TOKENS, 'raise MAX_MODEL_LEN'

llm = LLM(model=MODEL_ID, dtype='bfloat16', trust_remote_code=True,
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=GPU_MEM_UTIL,
          seed=SEED, enforce_eager=True,
          enable_lora=True, max_lora_rank=64)
lora_req = LoRARequest('sft', 1, ADAPTER_PATH)

INFO 05-31 13:58:15 [__init__.py:244] Automatically detected platform cuda.
VAL: 169 held-out rows


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

prompt tokens: max=3038 | headroom for gen = 37922


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

INFO 05-31 13:58:41 [config.py:841] This model supports multiple tasks: {'embed', 'reward', 'classify', 'generate'}. Defaulting to 'generate'.
INFO 05-31 13:58:41 [config.py:1472] Using max model len 40960
INFO 05-31 13:58:41 [config.py:2285] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 05-31 13:58:41 [cuda.py:102] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## 6. Generate val responses + score

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)                       # the dir that was missing
ckpt_name  = os.path.basename(ADAPTER_PATH.rstrip('/'))
preds_path = f'{OUTPUT_DIR}/{ckpt_name}_val_preds.jsonl'

# Resume: load whatever was already written so a re-run continues, not restarts.
done = {}
if os.path.exists(preds_path):
    for line in open(preds_path):
        r = json.loads(line); done[r['id']] = r
    print(f'resuming: {len(done)} already saved, {len(order) - len(done)} left')

sp = SamplingParams(temperature=0.6, top_p=0.95, top_k=20, min_p=0.0,
                    max_tokens=MAX_GEN_TOKENS)

CHUNK = 32
todo  = [(rid, p) for rid, p in zip(order, prompts) if rid not in done]

for i in range(0, len(todo), CHUNK):
    b_ids     = [rid for rid, _ in todo[i:i+CHUNK]]
    b_prompts = [p   for _,  p in todo[i:i+CHUNK]]
    outs = llm.generate(b_prompts, sp, lora_request=lora_req)
    with open(preds_path, 'a') as f:                         # append + close = flush to Drive
        for rid, out in zip(b_ids, outs):
            o = out.outputs[0]
            rec = {'id': rid, 'response': o.text,
                   'finish_reason': o.finish_reason,
                   'n_gen_tokens': len(o.token_ids),
                   'truncated': (o.finish_reason == 'length')}
            f.write(json.dumps(rec) + '\n')
            done[rid] = rec
    print(f'saved {min(i+CHUNK, len(todo))}/{len(todo)} new  (total {len(done)}/{len(order)})')

results = [done[rid] for rid in order]                        # in val order, for §7
n_trunc = sum(r['truncated'] for r in results)
print(f'done: {len(results)} | truncated: {n_trunc}')

Adding requests:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

saved 32/169 new  (total 32/169)


Adding requests:   0%|          | 0/32 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/32 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

saved 160/169 new  (total 160/169)


Adding requests:   0%|          | 0/9 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/9 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

saved 169/169 new  (total 169/169)
done: 169 | truncated: 2


## 7. Score vs Phase 1 baseline (71.60% MC 80.4 / FS 66.7 / FM 67.7)

In [ ]:
trunc_ids = {r['id'] for r in results if r['truncated']}
preds = [{'id': r['id'], 'response': r['response']} for r in results]
summary, per_row = H.score(preds, val_rows, truncated_ids=trunc_ids)

print(f'### ADAPTER: {ckpt_name} — VAL accuracy ###')
H.print_summary(summary)

BASELINE = {'mc': 0.8036, 'free_single': 0.6667,
            'free_multi': 0.6774, 'overall': 0.7160}
print()
print('Lift vs Phase 1 baseline:')
for b in ['mc', 'free_single', 'free_multi', 'overall']:
    new = summary[b]['acc']
    delta = (new - BASELINE[b]) * 100
    sign = '+' if delta >= 0 else ''
    print(f'  {b:<14} {new*100:6.2f}%  ({sign}{delta:.2f} pp vs {BASELINE[b]*100:.2f}%)')

### ADAPTER: qwen3_4b_thinking_sft_lora — VAL accuracy ###
EVALUATION RESULTS
  MC         :   41 /   56  ( 73.21%)
  Free-1blank:   32 /   51  ( 62.75%)
  Free-multi :    1 /   62  (  1.61%)
  Overall    :   74 /  169  ( 43.79%)
--------------------------------------------------------
  extract_fail (free-form, empty extract): 0
  MC scored via fallback regex          : 2
    ...of which graded CORRECT (fragile): 0
  generations truncated at token cap    : 2
  grading timed out (counted INCORRECT)  : 0

Lift vs Phase 1 baseline:
  mc              73.21%  (-7.15 pp vs 80.36%)
  free_single     62.75%  (-3.92 pp vs 66.67%)
  free_multi       1.61%  (-66.13 pp vs 67.74%)
  overall         43.79%  (-27.81 pp vs 71.60%)
